In [1]:
import wrds
import pandas as pd
import numpy as np
from typing import Optional, List, Tuple
import datetime


In [2]:
def get_unadjusted_prices(conn: wrds.Connection, 
                         permnos: Optional[List[int]] = None,
                         start_date: Optional[str] = None,
                         end_date: Optional[str] = None) -> pd.DataFrame:
    """
    Fetch unadjusted daily stock prices from CRSP.
    
    Args:
        conn: WRDS database connection
        permnos: List of PERMNOs to fetch. If None, fetches all.
        start_date: Start date in 'YYYY-MM-DD' format
        end_date: End date in 'YYYY-MM-DD' format
        
    Returns:
        DataFrame with columns: permno, dlycaldt, dlyprc, dlyvol, dlyret, dlybid, dlyask, shrout, dlycumfacpr, dlycumfacshr
    """
    query = """
    SELECT permno, dlycaldt as date, dlyprc as prc, dlyvol as vol, dlyret as ret, 
           dlybid as bid, dlyask as ask, shrout, dlycumfacpr as cfacpr, dlycumfacshr as cfacshr
    FROM crsp.dsf_v2
    WHERE 1=1
    """
    
    conditions = []
    if permnos is not None:
        permno_list = ','.join(map(str, permnos))
        conditions.append(f"permno IN ({permno_list})")
    
    if start_date is not None:
        conditions.append(f"dlycaldt >= '{start_date}'")
        
    if end_date is not None:
        conditions.append(f"dlycaldt <= '{end_date}'")
    
    if conditions:
        query += " AND " + " AND ".join(conditions)
    
    query += " ORDER BY permno, dlycaldt"
    
    return conn.raw_sql(query, date_cols=['date'])


In [3]:
def get_corporate_actions(conn: wrds.Connection,
                         permnos: Optional[List[int]] = None,
                         start_date: Optional[str] = None,
                         end_date: Optional[str] = None) -> pd.DataFrame:
    """
    Fetch corporate action data from the CRSP distributions table.

    Uses crsp.stkdistributions, the CIZ-format event table that pairs with
    crsp.dsf_v2.  The legacy SIZ table (crsp.dsedist) carries the same events
    under different column names and is no longer the maintained one.

    The single SIZ DISTCD is replaced by a family of narrower codes.  The two
    that matter for classification are DISTYPE and DISDETAILTYPE:

        CD  / CDIV    cash dividend (FACPR = 0)
        SD  / SDIV    special cash dividend (FACPR = 0)
        CG  / CAPG    capital gains distribution
        ROC / ROC     return of capital
        FRS / STKSPL  stock split, forward or reverse (FACPR != 0)
        FRS / STKDIV  stock dividend (FACPR != 0)
        SP  / SEC*    distribution of another security, e.g. a spinoff
        CP  / *       merger, liquidation or other terminal cash payment;
                      FACPR is ~-1.0 because the share line goes to zero

    DISORDINARYFLG ('Y'/'N') separates ordinary distributions from the rest,
    and DISTAXTYPE carries the tax character.  See the CRSP manual for the
    full code lists - the ones above cover the overwhelming majority of rows.

    Args:
        conn: WRDS database connection
        permnos: List of PERMNOs to fetch. If None, fetches all.
        start_date: Start date in 'YYYY-MM-DD' format
        end_date: End date in 'YYYY-MM-DD' format

    Returns:
        DataFrame with distribution events including dividends and splits.
        Columns are aliased to the SIZ names (ex_date, divamt, facpr, facshr,
        acperm, accomp) so the rest of this module is unchanged.
    """
    query = """
    SELECT permno, disexdt as ex_date, disrecorddt as record_date,
           dispaydt as pay_date, disdeclaredt as declare_date, disseqnbr as seq_nbr,
           distype, disdetailtype, disfreqtype, dispaymenttype, distaxtype,
           disordinaryflg, disorigcurtype,
           disdivamt as divamt, disfacpr as facpr, disfacshr as facshr,
           dispermno as acperm, dispermco as accomp
    FROM crsp.stkdistributions
    WHERE 1=1
    """

    conditions = []
    if permnos is not None:
        permno_list = ','.join(map(str, permnos))
        conditions.append(f"permno IN ({permno_list})")

    if start_date is not None:
        conditions.append(f"disexdt >= '{start_date}'")

    if end_date is not None:
        conditions.append(f"disexdt <= '{end_date}'")

    if conditions:
        query += " AND " + " AND ".join(conditions)

    # DISSEQNBR orders multiple distributions sharing one ex-date.
    query += " ORDER BY permno, disexdt, disseqnbr"

    dist = conn.raw_sql(query, date_cols=['ex_date', 'record_date', 'pay_date',
                                          'declare_date'])

    # DISDIVAMT is NULL on pure share-count events (SIZ wrote 0.0 there), and
    # FACPR/FACSHR are NULL on pure cash events.  Left as NaN on purpose:
    # calculate_adjustment_factors fills the factors, and a NaN dividend is a
    # more honest input to a yield calculation than a fabricated zero.
    return dist


In [3]:
def get_corporate_actions(conn: wrds.Connection,
                         permnos: Optional[List[int]] = None,
                         start_date: Optional[str] = None,
                         end_date: Optional[str] = None) -> pd.DataFrame:
    """
    Fetch corporate action data from the CRSP distributions table.

    Uses crsp.stkdistributions, the CIZ-format event table that pairs with
    crsp.dsf_v2.  The legacy SIZ table (crsp.dsedist) carries the same events
    under different column names and is no longer the maintained one.

    The single SIZ DISTCD is replaced by a family of narrower codes.  The two
    that matter for classification are DISTYPE and DISDETAILTYPE:

        CD  / CDIV    cash dividend (FACPR = 0)
        SD  / SDIV    special cash dividend (FACPR = 0)
        CG  / CAPG    capital gains distribution
        ROC / ROC     return of capital
        FRS / STKSPL  stock split, forward or reverse (FACPR != 0)
        FRS / STKDIV  stock dividend (FACPR != 0)
        SP  / SEC*    distribution of another security, e.g. a spinoff
        CP  / *       merger, liquidation or other terminal cash payment;
                      FACPR is ~-1.0 because the share line goes to zero

    DISORDINARYFLG ('Y'/'N') separates ordinary distributions from the rest,
    and DISTAXTYPE carries the tax character.  See the CRSP manual for the
    full code lists - the ones above cover the overwhelming majority of rows.

    Args:
        conn: WRDS database connection
        permnos: List of PERMNOs to fetch. If None, fetches all.
        start_date: Start date in 'YYYY-MM-DD' format
        end_date: End date in 'YYYY-MM-DD' format

    Returns:
        DataFrame with distribution events including dividends and splits.
        Columns are aliased to the SIZ names (ex_date, divamt, facpr, facshr,
        acperm, accomp) so the rest of this module is unchanged.
    """
    query = """
    SELECT permno, disexdt as ex_date, disrecorddt as record_date,
           dispaydt as pay_date, disdeclaredt as declare_date, disseqnbr as seq_nbr,
           distype, disdetailtype, disfreqtype, dispaymenttype, distaxtype,
           disordinaryflg, disorigcurtype,
           disdivamt as divamt, disfacpr as facpr, disfacshr as facshr,
           dispermno as acperm, dispermco as accomp
    FROM crsp.stkdistributions
    WHERE 1=1
    """

    conditions = []
    if permnos is not None:
        permno_list = ','.join(map(str, permnos))
        conditions.append(f"permno IN ({permno_list})")

    if start_date is not None:
        conditions.append(f"disexdt >= '{start_date}'")

    if end_date is not None:
        conditions.append(f"disexdt <= '{end_date}'")

    if conditions:
        query += " AND " + " AND ".join(conditions)

    # DISSEQNBR orders multiple distributions sharing one ex-date.
    query += " ORDER BY permno, disexdt, disseqnbr"

    dist = conn.raw_sql(query, date_cols=['ex_date', 'record_date', 'pay_date',
                                          'declare_date'])

    # DISDIVAMT is NULL on pure share-count events (SIZ wrote 0.0 there), and
    # FACPR/FACSHR are NULL on pure cash events.  Left as NaN on purpose:
    # calculate_adjustment_factors fills the factors, and a NaN dividend is a
    # more honest input to a yield calculation than a fabricated zero.
    return dist



In [8]:
def calculate_adjustment_factors(distributions: pd.DataFrame,
                                 prices: pd.DataFrame) -> pd.DataFrame:
    """
    Cumulative price adjustment factors for splits AND dividends.

    A split is multiplicative but a dividend is additive, so the two cannot be
    compounded side by side as they stand.  The standard fix is to divide the
    dividend by the CUM-DIVIDEND close - the last close strictly BEFORE the
    ex-date - which turns it into a multiplicative factor too:

        price_adj_factor = 1 / (1 + FACPR)
        div_adj_factor   = 1 - D / P(t-1)

    D / P(t-1) is a ratio of two unadjusted prices, so it is scale free: the
    two factors are independent and simply multiply.

    Check the arithmetic on a $1 dividend on a $100 stock that opens ex at $99.
    div_adj_factor = 1 - 1/100 = 0.99, and 99 / (100 * 0.99) - 1 = 0, a zero
    total return across the ex-date, which is right.  Dividing by P(t-1) + D
    instead is a common variant and is subtly wrong - it leaves a residual of a
    few basis points per event.

    Args:
        distributions: DataFrame from get_corporate_actions()
        prices: DataFrame from get_unadjusted_prices().  Needed for P(t-1), so
            it must cover the trading day before each ex-date.

    Returns:
        One row per (permno, ex_date), same-day events collapsed:

            divamt            cash per share (non-cash events contribute 0)
            prev_close        the P(t-1) above
            price_adj_factor  this event, splits only
            div_adj_factor    this event, dividends only
            cum_price_adj     compounded backward, splits only -> CRSP CFACPR
            cum_total_adj     compounded backward, splits AND dividends

        Both cum_ columns apply to every date STRICTLY BEFORE that row's
        ex_date.  Multiply prices by cum_price_adj for a split-adjusted series
        (real price levels, dividends still visible as drops on the ex-date),
        or by cum_total_adj for a total-return series.
    """
    CASH_DISTYPES = ('CD', 'SD', 'CG', 'ROC')

    cols = ['permno', 'ex_date', 'divamt', 'prev_close',
            'price_adj_factor', 'div_adj_factor',
            'cum_price_adj', 'cum_total_adj']

    if distributions.empty:
        return pd.DataFrame(columns=cols)

    dist = distributions.copy()
    dist['ex_date'] = pd.to_datetime(dist['ex_date']).dt.normalize()

    # CRSP states a distribution as a PROPORTIONAL change in share count:
    # FACPR = (s(t) - s(t')) / s(t').  A 2-for-1 split is 1.0, a 1-for-2
    # reverse split is -0.5, a cash dividend is 0.  So prices quoted before the
    # ex-date are divided by (1 + FACPR), never by FACPR, and cash dividends
    # drop out of that formula on their own: 1 + 0 = 1.
    dist['_mult'] = 1.0 + pd.to_numeric(dist['facpr'], errors='coerce').fillna(0.0)
    is_cash = dist['distype'].astype(str).str.strip().str.upper().isin(CASH_DISTYPES)
    dist['_div'] = (pd.to_numeric(dist['divamt'], errors='coerce')
                      .fillna(0.0).where(is_cash, 0.0))

    # Several distributions can share one ex-date.  Collapse them first: cash
    # amounts ADD, split factors COMPOUND, and they share one prior close.
    ev = (dist.groupby(['permno', 'ex_date'], as_index=False)
              .agg(divamt=('_div', 'sum'), mult=('_mult', 'prod')))

    # prev_close: the last close STRICTLY before the ex-date.  The ex-date's
    # own close is already ex-dividend, hence allow_exact_matches=False.
    # CRSP stores a NEGATIVE price when the close is a bid/ask average.
    px = prices[['permno', 'date', 'prc']].copy()
    px['date'] = pd.to_datetime(px['date']).dt.normalize()
    px['prc'] = px['prc'].abs()
    ev = pd.merge_asof(
        ev.sort_values('ex_date'),
        (px.dropna(subset=['prc']).sort_values('date')
           .rename(columns={'date': 'ex_date', 'prc': 'prev_close'})),
        on='ex_date', by='permno',
        direction='backward', allow_exact_matches=False)

    ev['price_adj_factor'] = np.where(ev['mult'] > 0, 1.0 / ev['mult'], 1.0)
    # A pure split pays no cash, so pin it at 1.0 rather than computing
    # 1 - 0/prev_close, which would be NaN whenever prev_close is missing.
    div_f = np.where(ev['divamt'] == 0.0, 1.0,
                     1.0 - ev['divamt'] / ev['prev_close'])
    # Non-positive means either no prior close, or a distribution worth the
    # whole security (a liquidation).  Neither is a usable price factor.
    ev['div_adj_factor'] = pd.Series(div_f, index=ev.index).where(lambda s: s > 0)

    # Compound backward from the end of the sample.
    ev = ev.sort_values(['permno', 'ex_date'], ascending=[True, False])
    ev['_div_f'] = ev['div_adj_factor'].fillna(1.0)
    grp = ev.groupby('permno', sort=False)
    ev['cum_price_adj'] = grp['price_adj_factor'].cumprod()
    ev['cum_total_adj'] = ev['cum_price_adj'] * grp['_div_f'].cumprod()

    return ev.sort_values(['permno', 'ex_date']).reset_index(drop=True)[cols]


In [9]:
conn = wrds.Connection(wrds_username='cpehlivanian')

Loading library list...
Done


In [10]:
start_date = '2002-01-01'
end_date = '2023-12-31'
permnos = [10107]

prices = get_unadjusted_prices(conn, permnos, start_date, end_date)
distributions = get_corporate_actions(conn, permnos, start_date, end_date)
adjustments = calculate_adjustment_factors(distributions, prices)

In [11]:
prices.head(10)

,permno,date,prc,vol,ret,bid,ask,shrout,cfacpr,cfacshr
0,10107,2002-01-02,67.04,24469510.0,0.011925,67.04,67.05,5403000,2.0,2.0
1,10107,2002-01-03,69.23,34106964.0,0.032667,69.23,69.23,5403000,2.0,2.0
2,10107,2002-01-04,68.9,26609046.0,-0.004767,68.89,68.92,5403000,2.0,2.0
3,10107,2002-01-07,68.56,29592070.0,-0.004935,68.55,68.56,5403000,2.0,2.0
4,10107,2002-01-08,69.38,24311452.0,0.01196,69.37,69.38,5403000,2.0,2.0
5,10107,2002-01-09,68.71,38449903.0,-0.009657,68.65,68.66,5403000,2.0,2.0
6,10107,2002-01-10,69.28,25486128.0,0.008296,69.29,69.33,5403000,2.0,2.0
7,10107,2002-01-11,68.61,36925365.0,-0.009671,68.61,68.64,5403000,2.0,2.0
8,10107,2002-01-14,68.47,25695919.0,-0.002041,68.43,68.44,5403000,2.0,2.0
9,10107,2002-01-15,69.55,30685268.0,0.015773,69.54,69.55,5403000,2.0,2.0


In [12]:
distributions.head(10)

,permno,ex_date,record_date,pay_date,declare_date,seq_nbr,distype,disdetailtype,disfreqtype,dispaymenttype,distaxtype,disordinaryflg,disorigcurtype,divamt,facpr,facshr,acperm,accomp
0,10107,2003-02-18,2003-01-27,2003-02-14,2003-01-16,1,FRS,STKSPL,N/A,SS,N,N,N/A,<NA>,1.0,1.0,0,0
1,10107,2003-02-19,2003-02-21,2003-03-07,2003-01-16,1,CD,CDIV,U,USD,D,Y,USD,0.08,0.0,0.0,0,0
2,10107,2003-10-15,2003-10-17,2003-11-07,2003-09-12,1,CD,CDIV,A,USD,D,Y,USD,0.16,0.0,0.0,0,0
3,10107,2004-08-23,2004-08-25,2004-09-14,2004-07-15,1,CD,CDIV,Q,USD,D,Y,USD,0.08,0.0,0.0,0,0
4,10107,2004-11-15,2004-11-17,2004-12-02,2004-09-15,1,CD,CDIV,Q,USD,D,Y,USD,0.08,0.0,0.0,0,0
5,10107,2004-11-15,2004-11-17,2004-12-02,2004-11-09,2,SD,SDIV,E,USD,D,Y,USD,3.0,0.0,0.0,0,0
6,10107,2005-02-15,2005-02-17,2005-03-10,2004-12-08,1,CD,CDIV,Q,USD,D,Y,USD,0.08,0.0,0.0,0,0
7,10107,2005-05-16,2005-05-18,2005-06-09,2005-03-23,1,CD,CDIV,Q,USD,D,Y,USD,0.08,0.0,0.0,0,0
8,10107,2005-08-15,2005-08-17,2005-09-08,2005-06-15,1,CD,CDIV,Q,USD,D,Y,USD,0.08,0.0,0.0,0,0
9,10107,2005-11-15,2005-11-17,2005-12-08,2005-09-23,1,CD,CDIV,Q,USD,D,Y,USD,0.08,0.0,0.0,0,0


In [13]:
adjustments.head(10)

,permno,ex_date,divamt,prev_close,price_adj_factor,div_adj_factor,cum_price_adj,cum_total_adj
0,10107,2003-02-18,0.0,48.3,0.5,1.000000,0.5,0.309981
1,10107,2003-02-19,0.08,24.96,1.0,0.996795,1.0,0.619962
2,10107,2003-10-15,0.16,28.68,1.0,0.994421,1.0,0.621955
3,10107,2004-08-23,0.08,27.2,1.0,0.997059,1.0,0.625445
4,10107,2004-11-15,3.08,29.97,1.0,0.897231,1.0,0.627290
5,10107,2005-02-15,0.08,26.01,1.0,0.996924,1.0,0.699140
6,10107,2005-05-16,0.08,25.3,1.0,0.996838,1.0,0.701297
7,10107,2005-08-15,0.08,27.05,1.0,0.997043,1.0,0.703521
8,10107,2005-11-15,0.08,27.35,1.0,0.997075,1.0,0.705608
9,10107,2006-02-15,0.09,26.65,1.0,0.996623,1.0,0.707678


In [14]:
conn.close()
